<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Generator Audiovisual with Cosmos Framework

This notebook runs Cosmos3 audiovisual generation through the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

Run all Cosmos3-Nano examples first, then run the Cosmos3-Super T2V/I2V examples without audio. Each section uses the matching checkpoint explicitly.


## 1. Prerequisites

Use a Linux machine with NVIDIA GPU access, model access on Hugging Face, and either `uvx hf@latest auth login` or `HF_TOKEN` set. Use cache/output paths with enough disk space.

> **Headless servers:** if you see an error like `libxcb.so.1: cannot open shared object file` (a missing system graphics library) when importing or running the model, install the required system libraries:
>
> ```bash
> apt-get install -y libxcb1 libgl1 libglib2.0-0
> ```

> **uv version:** this notebook needs `uv >= 0.11.3` (enforced by the framework's `pyproject.toml`). Older versions fail to parse the project config (e.g. the `[tool.uv.audit]` section) and do not recognize newer `--torch-backend` values such as `cu130`. If you hit version-related errors, upgrade with `uv self update` (or reinstall from https://astral.sh/uv).


## 2. Configure Paths and Environment

The defaults are relative to this `cosmos` checkout and use the CUDA 13 or 12.8 dependency group depending on the CUDA version installed on your system (`cu130-train` or `cu128-train`):

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-audiovisual
export COSMOS3_NUM_GPUS=4
export HF_HOME=/path/to/large/huggingface/cache
export CUDA_VISIBLE_DEVICES=0,1,2,3
```


In [ ]:
from pathlib import Path
import os
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
def default_framework_repo(root: Path) -> Path:
    candidates = [
        root / "packages" / "cosmos-framework",
        root / "packages" / "cosmos3",
    ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return candidates[0]


COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "git@github.com:NVIDIA/cosmos-framework.git")
COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = Path(os.environ.get("UV_PROJECT_ENVIRONMENT", COSMOS3_REPO / ".venv")).resolve()
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "4")
COSMOS3_AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
COSMOS3_AUDIOVISUAL_OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT", COSMOS3_AUDIOVISUAL_ROOT / "outputs" / "notebooks")
).resolve()

os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"] = str(COSMOS3_UV_ENV)
os.environ.setdefault("UV_PROJECT_ENVIRONMENT", str(COSMOS3_UV_ENV))
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"] = str(COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
os.environ.setdefault("UV_CACHE_DIR", str(Path.home() / ".cache" / "uv"))
os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,3")
os.environ.setdefault("COSMOS3_MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("COSMOS3_TEXT_MASTER_PORT", free_local_port())
os.environ.setdefault("COSMOS3_IMAGE_MASTER_PORT", free_local_port())

print(f"COSMOS_ROOT: {COSMOS_ROOT}")
for key in [
    "COSMOS3_REPO",
    "COSMOS3_GIT_URL",
    "COSMOS3_UV_GROUP",
    "COSMOS3_UV_ENV",
    "COSMOS3_NUM_GPUS",
    "COSMOS3_AUDIOVISUAL_OUTPUT_ROOT",
    "UV_CACHE_DIR",
    "UV_PROJECT_ENVIRONMENT",
    "HF_HOME",
    "CUDA_VISIBLE_DEVICES",
]:
    print(f"{key}: {os.environ[key]}")


## 3. Clone or Reuse Cosmos Framework


In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  echo "Set COSMOS3_REPO to the staged framework checkout, for example packages/cosmos-framework."
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
git status --short --branch
git remote -v


## 4. Install Native PyTorch Dependencies

Installs framework dependencies with `cu130-train` by default.


In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export GIT_LFS_SKIP_SMUDGE=1
cd "$COSMOS3_REPO"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
echo "Using UV_PROJECT_ENVIRONMENT=$UV_PROJECT_ENVIRONMENT"
uv sync --all-extras --group="$COSMOS3_UV_GROUP"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
  echo "Check UV_PROJECT_ENVIRONMENT above; uv created the environment somewhere else or the install failed."
  exit 1
fi


## 5. Verify GPU and Python Environment


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install Native PyTorch Dependencies cell first, or set UV_PROJECT_ENVIRONMENT/COSMOS3_UV_ENV to the uv environment path."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY


## 6. Preview Available Inputs


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

assets_dir = COSMOS3_AUDIOVISUAL_ROOT / "assets"
for prompt_dir in sorted((assets_dir / "prompts").iterdir()):
    if not prompt_dir.is_dir():
        continue
    print(f"{prompt_dir.relative_to(assets_dir)}:")
    for prompt_path in sorted(prompt_dir.glob("*.json")):
        data = json.loads(prompt_path.read_text())
        caption = (
            data.get("temporal_caption")
            or data.get("comprehensive_t2i_caption")
            or data.get("extra", {}).get("prompt", "")
        )
        print(f"  {prompt_path.name}: {caption[:180]}{'...' if len(caption) > 180 else ''}")
    print()

for image_dir in sorted((assets_dir / "images").iterdir()):
    if not image_dir.is_dir():
        continue
    print(f"{image_dir.relative_to(assets_dir)}:")
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            print(f"  {image_path.name}")
            display(Image(filename=str(image_path), width=420))


## 7. Define Asset Sets, Payload Helpers, and Viewer Helpers


In [ ]:
import json
import os
from pathlib import Path
from IPython.display import Image, display

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

FIXED_SAMPLING = {
    "num_steps": 35,
    "guidance": 6.0,
    "shift": 10.0,
    "fps": 24,
    "num_frames": 189,
    "resolution": "720",
    "aspect_ratio": "16,9",
    "seed": 0,
}

# All asset paths are repo-relative under cookbooks/cosmos3/generator/audiovisual.
# The Nano/Super cases without audio are intentionally separate so each run uses the matching model.
ASSET_SETS = {
    "t2i": {
        "model": "Cosmos3-Nano",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
        "enable_sound": False,
    },
    "t2i_super": {
        "model": "Cosmos3-Super",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
        "enable_sound": False,
    },
    "t2v_nano_noaudio": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
        "enable_sound": False,
    },
    "t2vs": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_pouring_water_audio.json",
        "enable_sound": True,
    },
    "i2v_nano_noaudio": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "image": "assets/images/image2video/car_driving.jpg",
        "enable_sound": False,
    },
    "i2vs": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/coastal_road_audio.json",
        "image": "assets/images/image2video/coastal_road_audio.jpg",
        "enable_sound": True,
    },
    "t2v_super_noaudio": {
        "model": "Cosmos3-Super",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
        "enable_sound": False,
    },
    "i2v_super_noaudio": {
        "model": "Cosmos3-Super",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "image": "assets/images/image2video/car_driving.jpg",
        "enable_sound": False,
    },
}


def asset_path(relative_path: str) -> Path:
    path = COSMOS3_AUDIOVISUAL_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path.resolve()


def compact_json_file(path: Path) -> str:
    return json.dumps(json.loads(path.read_text()), ensure_ascii=True, separators=(",", ":"))


def payload_dimensions(payload: dict) -> tuple[int, int]:
    if payload.get("resolution") == "720" and payload.get("aspect_ratio") == "16,9":
        return 720, 1280
    if payload.get("resolution") == "256" and payload.get("aspect_ratio") == "16,9":
        return 192, 320
    raise ValueError(f"Unsupported payload resolution/aspect ratio: {payload.get('resolution')} {payload.get('aspect_ratio')}")


def resolve_payload_path(payload_path: Path, value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return (payload_path.parent / path).resolve()


def create_payload(use_case: str, *, backend: str) -> tuple[Path, Path, str]:
    spec = ASSET_SETS[use_case]
    payload_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / "payloads" / use_case
    output_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / use_case
    payload_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    prompt_path = asset_path(spec["prompt"])
    negative_prompt = ""
    if spec["mode"] != "text2image":
        negative_prompt_path = asset_path(f"assets/negative_prompts/{spec['mode']}/neg_prompt.json")
        negative_prompt = compact_json_file(negative_prompt_path)
    payload_path = payload_dir / f"{use_case}.json"
    payload = {
        "model_mode": spec["mode"],
        "name": use_case,
        "prompt": compact_json_file(prompt_path),
        "negative_prompt": negative_prompt,
        "enable_sound": spec["enable_sound"],
        **FIXED_SAMPLING,
    }
    if spec["mode"] == "text2image":
        payload["num_frames"] = 1
    if spec["mode"] == "image2video":
        image_path = asset_path(spec["image"])
        payload["vision_path"] = os.path.relpath(image_path, payload_path.parent)

    payload_path.write_text(json.dumps(payload, indent=2) + "\n")

    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_INPUT"] = str(payload_path)
    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_OUTPUT"] = str(output_dir)

    print(f"model:   {spec['model']}")
    print(f"payload: {payload_path}")
    print(f"output:  {output_dir}")
    print(f"prompt:  {prompt_path.relative_to(COSMOS_ROOT)}")
    if "vision_path" in payload:
        image_display_path = resolve_payload_path(payload_path, payload["vision_path"])
        print(f"image:   {image_display_path.relative_to(COSMOS_ROOT)}")
        display(Image(filename=str(image_display_path), width=420))
    print(json.dumps({k: payload[k] for k in ["model_mode", "name", "enable_sound", "num_steps", "guidance", "shift", "fps", "num_frames", "resolution", "aspect_ratio", "seed"]}, indent=2))
    return payload_path, output_dir, spec["model"]


import base64
import html
from pathlib import Path
from IPython.display import HTML, display


def display_video(path: Path, *, width: int = 720) -> None:
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    label = html.escape(str(path))
    markup = f"""
<video controls playsinline preload="metadata" width="{width}" style="max-width: 100%; background: #000;">
  <source src="data:video/mp4;base64,{data}" type="video/mp4">
</video>
<div style="font-family: monospace; font-size: 12px; margin-top: 4px;">{label}</div>
"""
    display(HTML(markup))


def view_run(output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    videos = [
        path
        for path in sorted(output_dir.rglob("*.mp4"))
        if not path.name.endswith(("_preview.mp4", "_browser.mp4"))
    ]
    images = sorted(output_dir.rglob("*.png")) + sorted(output_dir.rglob("*.jpg")) + sorted(output_dir.rglob("*.jpeg"))
    if not videos and not images:
        print(f"No generated media found under {output_dir}")
        return
    for src in videos:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display_video(src)
    for src in images:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display(Image(filename=str(src), width=720))


## Use Cases

Run each use case top-to-bottom: create the JSON payload, run inference, then view the generated media. Nano examples come first; Super examples are last.


## Nano: Text to Image

Nano text-to-image generation using a structured JSON prompt.

### Create Payload

In [ ]:
t2i_payload, t2i_output, t2i_model = create_payload("t2i", backend="pytorch")

### Run

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_TEXT_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_T2I_INPUT" \
  -o "$COSMOS3_PYTORCH_T2I_OUTPUT" \
  --checkpoint-path "Cosmos3-Nano" \
  --seed=0

### View Results

In [ ]:
view_run(t2i_output)

## Super: Text to Image

Super text-to-image generation using the same structured JSON prompt.

### Create Payload

In [ ]:
t2i_super_payload, t2i_super_output, t2i_super_model = create_payload("t2i_super", backend="pytorch")

### Run

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_TEXT_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_T2I_SUPER_INPUT" \
  -o "$COSMOS3_PYTORCH_T2I_SUPER_OUTPUT" \
  --checkpoint-path "Cosmos3-Super" \
  --seed=0

### View Results

In [ ]:
view_run(t2i_super_output)

## Nano: Text to Video Without Audio

Nano text-to-video generation with audio disabled.

### Create Payload


In [ ]:
t2v_nano_noaudio_payload, t2v_nano_noaudio_output, t2v_nano_noaudio_model = create_payload("t2v_nano_noaudio", backend="pytorch")


### Run


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_TEXT_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_T2V_NANO_NOAUDIO_INPUT" \
  -o "$COSMOS3_PYTORCH_T2V_NANO_NOAUDIO_OUTPUT" \
  --checkpoint-path "Cosmos3-Nano" \
  --seed=0


### View Results


In [ ]:
view_run(t2v_nano_noaudio_output)


## Nano: Text to Video with Audio

Nano text-to-video generation with generated audio.

### Create Payload


In [ ]:
t2vs_payload, t2vs_output, t2vs_model = create_payload("t2vs", backend="pytorch")


### Run


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_TEXT_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_T2VS_INPUT" \
  -o "$COSMOS3_PYTORCH_T2VS_OUTPUT" \
  --checkpoint-path "Cosmos3-Nano" \
  --seed=0


### View Results


In [ ]:
view_run(t2vs_output)


## Nano: Image to Video Without Audio

Nano image-to-video generation using its paired image asset, with audio disabled.

### Create Payload


In [ ]:
i2v_nano_noaudio_payload, i2v_nano_noaudio_output, i2v_nano_noaudio_model = create_payload("i2v_nano_noaudio", backend="pytorch")


### Run


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_IMAGE_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_I2V_NANO_NOAUDIO_INPUT" \
  -o "$COSMOS3_PYTORCH_I2V_NANO_NOAUDIO_OUTPUT" \
  --checkpoint-path "Cosmos3-Nano" \
  --seed=0


### View Results


In [ ]:
view_run(i2v_nano_noaudio_output)


## Nano: Image to Video with Audio

Nano image-to-video generation using its paired image asset and generated audio.

### Create Payload


In [ ]:
i2vs_payload, i2vs_output, i2vs_model = create_payload("i2vs", backend="pytorch")


### Run


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_IMAGE_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_I2VS_INPUT" \
  -o "$COSMOS3_PYTORCH_I2VS_OUTPUT" \
  --checkpoint-path "Cosmos3-Nano" \
  --seed=0


### View Results


In [ ]:
view_run(i2vs_output)


## Super: Text to Video Without Audio

Super text-to-video generation with audio disabled.

### Create Payload


In [ ]:
t2v_super_noaudio_payload, t2v_super_noaudio_output, t2v_super_noaudio_model = create_payload("t2v_super_noaudio", backend="pytorch")


### Run


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_TEXT_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_T2V_SUPER_NOAUDIO_INPUT" \
  -o "$COSMOS3_PYTORCH_T2V_SUPER_NOAUDIO_OUTPUT" \
  --checkpoint-path "Cosmos3-Super" \
  --seed=0


### View Results


In [ ]:
view_run(t2v_super_noaudio_output)


## Super: Image to Video Without Audio

Super image-to-video generation using its paired image asset, with audio disabled.

### Create Payload


In [ ]:
i2v_super_noaudio_payload, i2v_super_noaudio_output, i2v_super_noaudio_model = create_payload("i2v_super_noaudio", backend="pytorch")


### Run


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_IMAGE_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$COSMOS3_PYTORCH_I2V_SUPER_NOAUDIO_INPUT" \
  -o "$COSMOS3_PYTORCH_I2V_SUPER_NOAUDIO_OUTPUT" \
  --checkpoint-path "Cosmos3-Super" \
  --seed=0


### View Results


In [ ]:
view_run(i2v_super_noaudio_output)
